<a href="https://colab.research.google.com/github/LuisGomes25/datasync-Projeto-CRISP-DM-abandono/blob/main/notebooks/04_avaliacao_implantacao.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os                                                        # funções do sistema: pastas e caminhos

REPO = "datasync-Projeto-CRISP-DM-abandono"                       # nome da pasta que o clone vai criar
URL = f"https://github.com/LuisGomes25/{REPO}.git"                # endereço público do repositório

if not os.path.exists(REPO) and os.path.basename(os.getcwd()) != REPO:   # só clona se ainda não clonou
    !git clone -q {URL}                                           # copia o repositório para o Colab (-q = sem barulho)

if os.path.basename(os.getcwd()) != REPO:                         # só entra na pasta se ainda não estiver nela
    os.chdir(REPO)                                                # muda a pasta de trabalho para o repositório

!pip install -q shap==0.54.1 reportlab                            # SHAP (versão do requirements.txt) e gerador de PDF

os.makedirs("resultados/04_avaliacao", exist_ok=True)
os.makedirs("resultados/05_implantacao", exist_ok=True)                  # cria a pasta de saída (não dá erro se já existir)
print("Pasta atual:", os.getcwd())                                # confirma onde estamos

In [ ]:
import numpy as np                                                # cálculos numéricos
import pandas as pd                                               # tabelas

SEED = 42                                                         # semente: garante o mesmo resultado em toda execução
ETAPAS = ["ensino_medio", "anos_finais", "anos_iniciais"]         # as três etapas, na ordem do notebook 03

MELHOR_MODELO = {                                                 # vencedores do notebook 03 (maior PR-AUC)
    "ensino_medio":  "Random_Forest",
    "anos_finais":   "Hist_Gradient_Boosting",
    "anos_iniciais": "Hist_Gradient_Boosting",
}

NUM_BASE = [                                                      # variáveis numéricas do cenário completo do notebook 03
    "tdi_distorcao", "atu_alunos_turma", "had_horas_aula",
    "dsu_doc_superior", "afd_formacao_adequada", "ied_esforco_alto",
    "icg_complexidade", "ird_regularidade", "inse_media",
    "flag_inse_ausente", "log_matriculas", "pct_noturno",
    "pct_integral", "pct_transporte_publico", "alunos_por_docente",
    "indice_infra", "flag_sem_turma_etapa",
]
CAT_BASE = ["SG_UF", "NO_CATEGORIA", "NO_DEPENDENCIA"]            # variáveis de texto: UF, urbana/rural, rede

df = pd.read_csv(
    "dados/tabela_analitica.csv",                                 # caminho relativo: funciona para qualquer pessoa
    encoding="utf-8-sig",                                         # remove um caractere invisível do início do arquivo
    dtype={"codigo_escola": str, "CO_MUNICIPIO": str},            # códigos como texto (não perdem zeros nem viram número)
    low_memory=False,                                             # lê o arquivo inteiro de uma vez para acertar os tipos
)

assert df.shape == (136713, 27), df.shape                         # trava: para tudo se a base não for a esperada
faltando = [c for c in NUM_BASE + CAT_BASE if c not in df.columns]   # lista colunas que deveriam existir e não existem
assert not faltando, faltando                                     # trava: para tudo se faltar alguma

def criar_alvo(serie, percentil=80):                              # mesma regra do notebook 03
    limite = float(serie.quantile(percentil / 100))               # valor da taxa no percentil pedido
    if limite == 0.0:                                             # nos Anos Iniciais o P80 é zero...
        return (serie > 0).astype(int), limite                    # ...então "alto" = qualquer abandono
    return (serie >= limite).astype(int), limite                  # demais etapas: taxa no limite ou acima

for etapa in ETAPAS:                                              # confere cada etapa
    sub = df[df["etapa"] == etapa]                                # separa as linhas da etapa
    y, lim = criar_alvo(sub["taxa_abandono"])                     # cria o alvo
    print(f"{etapa:15s} n={len(sub):>6,}  limite={lim:.2f}  alto={y.mean()*100:.2f}%")

In [ ]:
from sklearn.compose import ColumnTransformer                     # aplica tratamentos diferentes por tipo de coluna
from sklearn.pipeline import Pipeline                             # encadeia etapas: tratar dados → modelo
from sklearn.impute import SimpleImputer                          # preenche valores vazios
from sklearn.preprocessing import StandardScaler, OneHotEncoder   # padroniza números / transforma texto em colunas 0-1
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier   # os dois modelos vencedores
from sklearn.model_selection import StratifiedGroupKFold          # divide os dados mantendo municípios inteiros juntos
from sklearn.metrics import average_precision_score, roc_auc_score   # métricas do notebook 03 (PR-AUC e ROC-AUC)

def montar_modelo(nome):                                          # monta o mesmo pipeline do notebook 03
    num = Pipeline([
        ("imputacao", SimpleImputer(strategy="median")),          # vazio numérico → mediana (calculada só no treino)
        ("escala", StandardScaler()),                             # deixa as variáveis na mesma escala
    ])
    cat = Pipeline([
        ("imputacao", SimpleImputer(strategy="most_frequent")),   # vazio de texto → valor mais comum
        ("onehot", OneHotEncoder(handle_unknown="ignore",         # "SP" vira uma coluna 0/1; UF nova é ignorada
                                 sparse_output=False)),
    ])
    prep = ColumnTransformer([("num", num, NUM_BASE),             # aplica o tratamento numérico às numéricas
                              ("cat", cat, CAT_BASE)])            # e o de texto às categóricas

    if nome == "Random_Forest":                                   # mesmos parâmetros do notebook 03
        clf = RandomForestClassifier(n_estimators=250, max_depth=14, min_samples_leaf=8,
                                     class_weight="balanced", random_state=SEED, n_jobs=-1)
    else:
        clf = HistGradientBoostingClassifier(max_iter=250, max_depth=6, learning_rate=0.05,
                                             min_samples_leaf=20, class_weight="balanced",
                                             random_state=SEED)
    return Pipeline([("prep", prep), ("clf", clf)])               # tratamento + modelo num objeto só

dados, modelos, oof = {}, {}, {}                                  # guardam, por etapa: dados, modelo final e probabilidades

for etapa in ETAPAS:
    sub = (df[df["etapa"] == etapa]                               # linhas da etapa
           .dropna(subset=["taxa_abandono", "CO_MUNICIPIO"])      # exige taxa e município (igual ao notebook 03)
           .reset_index(drop=True))                               # renumera as linhas de 0 a n
    y, _ = criar_alvo(sub["taxa_abandono"])                       # alvo P80
    y = y.values                                                  # vira array simples
    X = sub[NUM_BASE + CAT_BASE]                                  # só as variáveis permitidas (sem a coluna-gabarito)
    grupos = sub["CO_MUNICIPIO"].values                           # município de cada escola

    prob = np.zeros(len(sub))                                     # espaço para a probabilidade de cada escola
    for treino, teste in StratifiedGroupKFold(n_splits=5).split(X, y, grupos):   # 5 rodadas
        m = montar_modelo(MELHOR_MODELO[etapa])                   # modelo novo a cada rodada
        m.fit(X.iloc[treino], y[treino])                          # aprende com 4/5 dos municípios
        prob[teste] = m.predict_proba(X.iloc[teste])[:, 1]        # dá nota ao 1/5 que ele nunca viu

    final = montar_modelo(MELHOR_MODELO[etapa])                   # modelo final...
    final.fit(X, y)                                               # ...treinado com tudo (será usado no SHAP)

    dados[etapa], modelos[etapa], oof[etapa] = (sub, X, y), final, prob
    print(f"{etapa:15s} PR-AUC={average_precision_score(y, prob):.4f}  "
          f"ROC-AUC={roc_auc_score(y, prob):.4f}")

In [ ]:
linhas = []                                                       # cada linha = uma etapa × um tamanho de lista

for etapa in ETAPAS:
    sub, X, y = dados[etapa]                                      # dados da etapa (célula 3)
    ordem = np.argsort(-oof[etapa])                               # posições das escolas da MAIOR para a menor probabilidade
    total_alto = y.sum()                                          # quantas escolas de alto abandono existem na etapa

    for k in [0.05, 0.10, 0.20]:                                  # capacidade da equipe: 5%, 10% ou 20% das escolas
        n_top = int(round(len(y) * k))                            # quantas escolas cabem na lista
        top = y[ordem[:n_top]]                                    # alvo real (0/1) das escolas do topo da lista
        linhas.append({
            "etapa": etapa,
            "k_pct": int(k * 100),                                # tamanho da lista em %
            "escolas_na_lista": n_top,
            "acertos": int(top.sum()),                            # quantas da lista são de alto abandono
            "precisao": round(top.mean(), 3),                     # acertos ÷ tamanho da lista
            "recall": round(top.sum() / total_alto, 3),           # acertos ÷ total de escolas de alto abandono
            "acaso": round(y.mean(), 3),                          # precisão se a lista fosse sorteada
            "vezes_melhor_que_acaso": round(top.mean() / y.mean(), 1),
        })

recall_k = pd.DataFrame(linhas)                                   # junta tudo numa tabela
recall_k.to_csv("resultados/04_avaliacao/01_recall_at_k.csv", index=False, encoding="utf-8-sig")   # salva para o relatório
recall_k

In [ ]:
import matplotlib.pyplot as plt                                   # gráficos
from sklearn.linear_model import LogisticRegression               # usada como "régua" para corrigir as probabilidades
from sklearn.metrics import brier_score_loss                      # erro médio das probabilidades (quanto menor, melhor)
from sklearn.calibration import calibration_curve                 # compara probabilidade prevista × proporção real

def logit(p):                                                     # transforma probabilidade (0 a 1) numa escala sem limites
    p = np.clip(p, 1e-6, 1 - 1e-6)                                # evita 0 e 1 exatos (dariam infinito)
    return np.log(p / (1 - p)).reshape(-1, 1)                     # fórmula do logit, no formato de coluna

prob_cal, calibradores, linhas = {}, {}, []                       # probabilidades corrigidas, réguas finais e tabela
fig, eixos = plt.subplots(1, 3, figsize=(15, 4.5))                # 3 gráficos lado a lado, um por etapa

for i, etapa in enumerate(ETAPAS):
    sub, X, y = dados[etapa]                                      # dados da etapa
    p = oof[etapa]                                                # probabilidades "honestas" da célula 3
    grupos = sub["CO_MUNICIPIO"].values                           # municípios

    p_cal = np.zeros(len(p))                                      # espaço para as probabilidades corrigidas
    for treino, teste in StratifiedGroupKFold(n_splits=5).split(X, y, grupos):
        regua = LogisticRegression()                              # régua nova a cada rodada
        regua.fit(logit(p[treino]), y[treino])                    # aprende "quanto encolher" em 4/5 dos municípios
        p_cal[teste] = regua.predict_proba(logit(p[teste]))[:, 1] # aplica no 1/5 restante

    calibradores[etapa] = LogisticRegression().fit(logit(p), y)   # régua final (para usar com o modelo final)
    prob_cal[etapa] = p_cal

    n = int(round(len(y) * 0.10))                                 # tamanho da lista dos 10%
    linhas.append({
        "etapa": etapa,
        "prevalencia_real": round(y.mean(), 3),                   # % real de alto abandono
        "prob_media_antes": round(p.mean(), 3),                   # média prevista sem correção
        "prob_media_depois": round(p_cal.mean(), 3),              # média prevista com correção
        "brier_antes": round(brier_score_loss(y, p), 4),
        "brier_depois": round(brier_score_loss(y, p_cal), 4),
        "precisao_top10_antes": round(y[np.argsort(-p)[:n]].mean(), 3),       # prova de que a lista
        "precisao_top10_depois": round(y[np.argsort(-p_cal)[:n]].mean(), 3),  # quase não muda
    })

    for nome, pp in [("Antes", p), ("Depois", p_cal)]:            # desenha as duas curvas
        real, previsto = calibration_curve(y, pp, n_bins=10, strategy="quantile")   # 10 faixas de mesmo tamanho
        eixos[i].plot(previsto, real, marker="o", label=nome)
    eixos[i].plot([0, 1], [0, 1], "--", color="gray", label="Perfeito")   # diagonal = calibração perfeita
    eixos[i].set_title(etapa)
    eixos[i].set_xlabel("Probabilidade prevista")
    eixos[i].set_ylabel("Proporção real")
    eixos[i].legend()

plt.tight_layout()
plt.savefig("resultados/04_avaliacao/02_calibracao.png", dpi=300, bbox_inches="tight")   # salva para o relatório
plt.show()

calibracao = pd.DataFrame(linhas)
calibracao.to_csv("resultados/04_avaliacao/02_calibracao.csv", index=False, encoding="utf-8-sig")
calibracao

In [ ]:
import shap                                                       # biblioteca de explicação dos modelos

resumo_shap = []                                                  # tabela final: importância e direção por variável
fig, eixos = plt.subplots(1, 3, figsize=(16, 5))                  # 3 gráficos de barras, um por etapa

for i, etapa in enumerate(ETAPAS):
    sub, X, y = dados[etapa]
    amostra = X.sample(n=2000, random_state=SEED)                 # 2.000 escolas sorteadas (sempre as mesmas)

    prep = modelos[etapa].named_steps["prep"]                     # parte de tratamento do modelo final
    clf = modelos[etapa].named_steps["clf"]                       # parte do modelo em si
    nomes = [n.split("__", 1)[1] for n in prep.get_feature_names_out()]   # "num__tdi_distorcao" → "tdi_distorcao"
    Xt = pd.DataFrame(prep.transform(amostra), columns=nomes)     # dados já tratados, do jeito que o modelo vê

    explicador = shap.TreeExplainer(clf)                          # explicador próprio para modelos de árvore
    valores = explicador.shap_values(Xt, check_additivity=False)  # contribuição de cada variável em cada escola
    if isinstance(valores, list):                                 # versões antigas devolvem uma lista por classe...
        valores = valores[1]                                      # ...pegamos a classe 1 (alto abandono)
    valores = np.asarray(valores)
    if valores.ndim == 3:                                         # Random Forest devolve [escola, variável, classe]...
        valores = valores[:, :, 1]                                # ...pegamos a classe 1

    # O OneHot transformou UF em 27 colunas (SG_UF_SP, SG_UF_AC...). Somamos de volta numa só.
    origem = {n: next((c for c in CAT_BASE if n.startswith(c + "_")), n) for n in nomes}   # coluna → variável original
    shap_df = pd.DataFrame(valores, columns=nomes)
    shap_agrup = shap_df.T.groupby(origem).sum().T                # soma as colunas de cada variável original

    importancia = shap_agrup.abs().mean().sort_values(ascending=False)   # média do impacto (sem sinal) = importância

    for var, imp in importancia.items():
        if var in NUM_BASE:                                       # numéricas: dá para calcular a direção
            rho = pd.Series(Xt[var]).corr(shap_df[var], method="spearman")   # valor alto → empurra para cima ou baixo?
            if pd.isna(rho):                                      # variável constante na amostra
                direcao = "sem variação na amostra"
            elif rho > 0:
                direcao = "maior valor → maior prob."
            else:
                direcao = "maior valor → menor prob."
        else:                                                     # categóricas (UF, urbana/rural, rede)
            direcao = "categórica (ver por categoria)"
        resumo_shap.append({"etapa": etapa, "variavel": var,
                            "importancia_media_abs": round(imp, 4), "direcao": direcao})

    top10 = importancia.head(10)[::-1]                            # 10 mais importantes (invertido para o gráfico)
    eixos[i].barh(top10.index, top10.values)
    eixos[i].set_title(f"{etapa} ({MELHOR_MODELO[etapa]})")
    eixos[i].set_xlabel("Impacto médio |SHAP|")

plt.tight_layout()
plt.savefig("resultados/04_avaliacao/03_shap_importancia.png", dpi=300, bbox_inches="tight")
plt.show()

shap_tabela = pd.DataFrame(resumo_shap)
shap_tabela.to_csv("resultados/04_avaliacao/03_shap_importancia.csv", index=False, encoding="utf-8-sig")
shap_tabela.groupby("etapa").head(8)                              # mostra as 8 principais de cada etapa

In [ ]:
shap_tabela[shap_tabela["direcao"] == "sem variação na amostra"]   # mostra só as variáveis constantes

In [ ]:
def alvo_percentil(serie, pct):                                   # mesma regra de antes, agora com o percentil como parâmetro
    limite = float(serie.quantile(pct / 100))                     # taxa de abandono no percentil pedido
    if limite == 0.0:                                             # nos Anos Iniciais P75 e P80 têm limite zero
        return (serie > 0).astype(int), limite                    # então "alto" = qualquer abandono
    return (serie >= limite).astype(int), limite                  # nas demais: taxa no limite ou acima

linhas = []                                                       # cada linha = uma etapa × um percentil
for etapa in ETAPAS:
    sub = (df[df["etapa"] == etapa]
           .dropna(subset=["taxa_abandono", "CO_MUNICIPIO"])
           .reset_index(drop=True))
    X = sub[NUM_BASE + CAT_BASE]                                  # mesmas variáveis da célula 3
    grupos = sub["CO_MUNICIPIO"].values                           # municípios (validação agrupada)

    for pct in [75, 80, 90]:                                      # os três cortes avaliados na definição do alvo
        y, lim = alvo_percentil(sub["taxa_abandono"], pct)        # alvo do percentil
        y = y.values
        prob = np.zeros(len(sub))                                 # probabilidades "honestas" (fora da dobra)
        for treino, teste in StratifiedGroupKFold(n_splits=5).split(X, y, grupos):
            m = montar_modelo(MELHOR_MODELO[etapa])               # mesmo modelo vencedor da etapa
            m.fit(X.iloc[treino], y[treino])
            prob[teste] = m.predict_proba(X.iloc[teste])[:, 1]

        n = int(round(len(y) * 0.10))                             # lista dos 10% do topo
        topo = y[np.argsort(-prob)[:n]]                           # alvo real das escolas do topo
        pr = average_precision_score(y, prob)
        linhas.append({
            "etapa": etapa, "percentil": pct, "limite": round(lim, 2),
            "prevalencia": round(y.mean(), 3),
            "pr_auc": round(pr, 4),
            "pr_auc_vezes_acaso": round(pr / y.mean(), 2),        # PR-AUC ÷ prevalência: comparável entre cortes
            "roc_auc": round(roc_auc_score(y, prob), 4),
            "precisao_top10": round(topo.mean(), 3),
            "vezes_acaso_top10": round(topo.mean() / y.mean(), 1),
        })

sensibilidade = pd.DataFrame(linhas)
sensibilidade.to_csv("resultados/04_avaliacao/04_sensibilidade_percentis.csv", index=False, encoding="utf-8-sig")
sensibilidade

In [ ]:
etapa = "anos_iniciais"
sub = df[df["etapa"] == etapa].dropna(subset=["taxa_abandono", "CO_MUNICIPIO"]).reset_index(drop=True)
X = sub[NUM_BASE + CAT_BASE]

ranking = {}                                                      # guarda a importância das variáveis em cada corte
for pct in [80, 90]:                                              # P80 (limite 0) × P90 (limite 0,6)
    y, _ = alvo_percentil(sub["taxa_abandono"], pct)
    modelo = montar_modelo(MELHOR_MODELO[etapa]).fit(X, y.values) # modelo final do corte

    amostra = X.sample(n=2000, random_state=SEED)                 # mesma amostra de 2.000 escolas da célula 6
    prep, clf = modelo.named_steps["prep"], modelo.named_steps["clf"]
    nomes = [n.split("__", 1)[1] for n in prep.get_feature_names_out()]
    Xt = pd.DataFrame(prep.transform(amostra), columns=nomes)

    valores = shap.TreeExplainer(clf).shap_values(Xt, check_additivity=False)
    valores = np.asarray(valores[1] if isinstance(valores, list) else valores)
    if valores.ndim == 3:
        valores = valores[:, :, 1]

    origem = {n: next((c for c in CAT_BASE if n.startswith(c + "_")), n) for n in nomes}
    agrup = pd.DataFrame(valores, columns=nomes).T.groupby(origem).sum().T   # soma as colunas de UF, rede etc.
    ranking[f"P{pct}"] = agrup.abs().mean().sort_values(ascending=False)

comparacao = pd.DataFrame({
    "importancia_P80": ranking["P80"], "posicao_P80": ranking["P80"].rank(ascending=False).astype(int),
    "importancia_P90": ranking["P90"], "posicao_P90": ranking["P90"].rank(ascending=False).astype(int),
}).sort_values("posicao_P90")
comparacao.round(4).to_csv("resultados/04_avaliacao/05_shap_ai_p80_vs_p90.csv", encoding="utf-8-sig")
comparacao.head(8).round(4)

In [ ]:
VARS_PERFIL = [                                                   # variáveis mais relevantes no SHAP e na EDA
    "tdi_distorcao", "inse_media", "log_matriculas", "pct_noturno",
    "pct_integral", "afd_formacao_adequada", "indice_infra",
    "pct_transporte_publico", "icg_complexidade",
]

linhas_num, linhas_cat = [], []                                   # tabelas de numéricas e de categóricas
for etapa in ETAPAS:
    sub, X, y = dados[etapa]                                      # dados da célula 3
    n = int(round(len(sub) * 0.10))                               # tamanho da lista dos 10%
    corte = np.sort(oof[etapa])[::-1][n - 1]                      # probabilidade da última escola que entra na lista
    sub = sub.assign(grupo=np.where(oof[etapa] >= corte,          # marca cada escola: prioritária ou não
                                    "prioritarias_top10", "demais"))

    med = sub.groupby("grupo")[VARS_PERFIL].median().T            # mediana de cada variável nos dois grupos
    for var in VARS_PERFIL:
        linhas_num.append({"etapa": etapa, "variavel": var,
                           "mediana_prioritarias": round(med.loc[var, "prioritarias_top10"], 2),
                           "mediana_demais": round(med.loc[var, "demais"], 2)})

    for col in ["NO_CATEGORIA", "NO_DEPENDENCIA"]:                # urbana/rural e rede
        tab = (pd.crosstab(sub[col], sub["grupo"], normalize="columns")   # % de cada categoria dentro de cada grupo
                 .mul(100).round(1))
        for cat in tab.index:
            linhas_cat.append({"etapa": etapa, "variavel": col, "categoria": cat,
                               "pct_prioritarias": tab.loc[cat, "prioritarias_top10"],
                               "pct_demais": tab.loc[cat, "demais"]})

perfil_num = pd.DataFrame(linhas_num)
perfil_cat = pd.DataFrame(linhas_cat)
perfil_num.to_csv("resultados/04_avaliacao/06_perfil_numericas.csv", index=False, encoding="utf-8-sig")
perfil_cat.to_csv("resultados/04_avaliacao/07_perfil_categoricas.csv", index=False, encoding="utf-8-sig")
display(perfil_num, perfil_cat)                                   # mostra as duas tabelas

In [ ]:
linhas_uf = []
for etapa in ETAPAS:
    sub, X, y = dados[etapa]
    n = int(round(len(sub) * 0.10))                               # tamanho da lista dos 10%
    corte = np.sort(oof[etapa])[::-1][n - 1]                      # probabilidade da última escola da lista
    sub = sub.assign(alvo=y,                                      # alto abandono real (0/1)
                     prioritaria=(oof[etapa] >= corte).astype(int))   # está na lista? (0/1)

    g = (sub.groupby("SG_UF")
            .agg(escolas=("alvo", "size"),                        # escolas da etapa na UF
                 alto_abandono=("alvo", "sum"),                   # quantas têm alto abandono real
                 prioritarias=("prioritaria", "sum"))             # quantas entraram na lista
            .reset_index())
    g["pct_escolas_da_etapa"] = (g.escolas / g.escolas.sum() * 100).round(1)            # peso da UF no total
    g["pct_das_prioritarias"] = (g.prioritarias / g.prioritarias.sum() * 100).round(1)  # peso da UF na lista
    g["razao_representacao"] = (g.pct_das_prioritarias / g.pct_escolas_da_etapa).round(2)   # >1 = acima do esperado
    g["pct_alto_abandono_real"] = (g.alto_abandono / g.escolas * 100).round(1)          # taxa real, sem modelo
    g.insert(0, "etapa", etapa)
    linhas_uf.append(g)

geo_uf = pd.concat(linhas_uf, ignore_index=True)
geo_uf.to_csv("resultados/04_avaliacao/08_geografia_uf.csv", index=False, encoding="utf-8-sig")

# Urbana × rural: % real de alto abandono (coluna NO_CATEGORIA)
loc = pd.concat([
    dados[e][0].assign(alvo=dados[e][2]).groupby("NO_CATEGORIA")["alvo"].mean().mul(100).round(1).rename(e)
    for e in ETAPAS], axis=1)
loc.to_csv("resultados/04_avaliacao/09_alto_abandono_urbana_rural.csv", encoding="utf-8-sig")

display(geo_uf.sort_values(["etapa", "razao_representacao"], ascending=[True, False]).groupby("etapa").head(5), loc)

In [ ]:
rk = pd.read_csv("resultados/04_avaliacao/01_recall_at_k.csv")             # tabela da célula 4
top10 = rk[rk["k_pct"] == 10][["etapa", "precisao", "acaso", "vezes_melhor_que_acaso"]].copy()
top10["meta"] = 2.0                                                   # meta: 2 vezes o acaso
top10["atende"] = top10["vezes_melhor_que_acaso"] >= top10["meta"]
top10

In [ ]:
# CÉLULA 13 · Overfitting (treino x validação), variação entre dobras,
#             acurácia e estabilidade das variáveis (permutação)
# Tempo: ~2 min
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.inspection import permutation_importance

linhas, imp_linhas = [], []                                        # métricas por dobra / importância por dobra
for etapa in ETAPAS:
    sub, X, y = dados[etapa]                                       # mesmos dados da célula 3
    grupos = sub["CO_MUNICIPIO"].values
    for dobra, (treino, teste) in enumerate(
            StratifiedGroupKFold(n_splits=5).split(X, y, grupos), 1):    # dobra = 1..5
        m = montar_modelo(MELHOR_MODELO[etapa])
        m.fit(X.iloc[treino], y[treino])

        for parte, idx in [("treino", treino), ("validacao", teste)]:   # mede nas duas partes
            p = m.predict_proba(X.iloc[idx])[:, 1]                       # probabilidade
            pred = (p >= 0.5).astype(int)                                # classe (corte 0,5, só para acurácia/F1)
            linhas.append({
                "etapa": etapa, "dobra": dobra, "parte": parte,
                "pr_auc": average_precision_score(y[idx], p),
                "roc_auc": roc_auc_score(y[idx], p),
                "acuracia": accuracy_score(y[idx], pred),
                "acuracia_balanceada": balanced_accuracy_score(y[idx], pred),
                "f1": f1_score(y[idx], pred),
            })

        # importância por permutação: embaralha uma variável e mede quanto a PR-AUC cai
        amostra = np.random.RandomState(SEED).choice(teste, size=min(3000, len(teste)), replace=False)
        pi = permutation_importance(m, X.iloc[amostra], y[amostra], scoring="average_precision",
                                    n_repeats=3, random_state=SEED, n_jobs=-1)
        for var, val in zip(X.columns, pi.importances_mean):
            imp_linhas.append({"etapa": etapa, "dobra": dobra, "variavel": var, "importancia": val})

dobras = pd.DataFrame(linhas)
resumo_dobras = (dobras.groupby(["etapa", "parte"])
                       [["pr_auc", "roc_auc", "acuracia", "acuracia_balanceada", "f1"]]
                       .agg(["mean", "std"]).round(3))             # média e desvio-padrão entre as 5 dobras
dobras.to_csv("resultados/04_avaliacao/14_metricas_por_dobra.csv", index=False, encoding="utf-8-sig")
resumo_dobras.to_csv("resultados/04_avaliacao/15_overfitting_treino_validacao.csv", encoding="utf-8-sig")

imp = pd.DataFrame(imp_linhas)
imp["posicao"] = imp.groupby(["etapa", "dobra"])["importancia"].rank(ascending=False)   # 1 = mais importante
estabilidade = (imp.groupby(["etapa", "variavel"])
                   .agg(importancia_media=("importancia", "mean"),
                        posicao_media=("posicao", "mean"),
                        posicao_min=("posicao", "min"),
                        posicao_max=("posicao", "max"))
                   .reset_index().sort_values(["etapa", "posicao_media"]).round(3))
estabilidade.to_csv("resultados/04_avaliacao/16_estabilidade_importancia.csv", index=False, encoding="utf-8-sig")

display(resumo_dobras, estabilidade.groupby("etapa").head(5))


In [ ]:
# CÉLULA 14 · Análise de erros (seção 5.3): quem o modelo erra?
VARS_ERRO = ["tdi_distorcao", "inse_media", "log_matriculas", "pct_noturno",
             "indice_infra", "afd_formacao_adequada", "taxa_abandono"]
partes = []
for etapa in ETAPAS:
    sub, X, y = dados[etapa]
    n = int(round(len(y) * 0.10))                                  # lista dos 10%
    na_lista = oof[etapa] >= np.sort(oof[etapa])[::-1][n - 1]      # True = escola entrou na lista
    tipo = np.select(
        [na_lista & (y == 1), na_lista & (y == 0), ~na_lista & (y == 1)],
        ["acerto_na_lista", "falso_positivo", "falso_negativo"],
        "fora_e_baixo")                                            # 4 grupos possíveis
    s = sub.assign(tipo=tipo, rural=(sub["NO_CATEGORIA"] == "Rural") * 100)
    g = s.groupby("tipo")[VARS_ERRO].median().round(2)             # mediana de cada grupo
    g["n"] = s["tipo"].value_counts()
    g["pct_rural"] = s.groupby("tipo")["rural"].mean().round(1)
    g.insert(0, "etapa", etapa)
    partes.append(g.reset_index())

erros = pd.concat(partes, ignore_index=True)
erros.to_csv("resultados/04_avaliacao/17_analise_erros.csv", index=False, encoding="utf-8-sig")
erros


In [ ]:
# CÉLULA 15 · Município (pergunta 8), cálculo corrigido
# Numerador e denominador contam as mesmas linhas escola x etapa.
partes = []
for etapa in ETAPAS:
    sub, X, y = dados[etapa]
    n = int(round(len(y) * 0.10))
    s = sub.assign(alvo=y, prioritaria=(oof[etapa] >= np.sort(oof[etapa])[::-1][n - 1]).astype(int))
    g = (s.groupby(["SG_UF", "CO_MUNICIPIO"])
           .agg(escolas=("alvo", "size"),
                alto_abandono=("alvo", "sum"),
                prioritarias=("prioritaria", "sum"))
           .reset_index())
    g["pct_alto_abandono_real"] = (g["alto_abandono"] / g["escolas"] * 100).round(1)
    g["pct_prioritarias"] = (g["prioritarias"] / g["escolas"] * 100).round(1)
    g.insert(0, "etapa", etapa)
    partes.append(g)

municipios = pd.concat(partes, ignore_index=True)
municipios.to_csv("resultados/04_avaliacao/18_municipios.csv", index=False, encoding="utf-8-sig")
assert (municipios["pct_alto_abandono_real"] <= 100).all()        # trava: nenhum % acima de 100

(municipios[municipios["escolas"] >= 20]                           # só municípios com ao menos 20 escolas
    .sort_values(["etapa", "pct_alto_abandono_real"], ascending=[True, False])
    .groupby("etapa").head(5))


In [ ]:
# CÉLULA 16 · Implantação (fase 6): lista priorizada com probabilidade calibrada
import os
from sklearn.linear_model import LogisticRegression
os.makedirs("resultados/05_implantacao", exist_ok=True)

def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p)).reshape(-1, 1)

partes = []
for etapa in ETAPAS:
    sub, X, y = dados[etapa]
    regua = LogisticRegression().fit(logit(oof[etapa]), y)         # calibração (mesma ideia da célula 5)
    prob = regua.predict_proba(logit(oof[etapa]))[:, 1]
    s = sub[["codigo_escola", "SG_UF", "CO_MUNICIPIO", "NO_DEPENDENCIA", "NO_CATEGORIA"]].assign(
        etapa=etapa, prob_calibrada=prob.round(3))
    s["faixa_prioridade"] = pd.qcut(                               # faixas pela posição na lista
        s["prob_calibrada"].rank(method="first", ascending=False),
        [0, .05, .10, .20, 1.0],
        labels=["1 (top 5%)", "2 (5-10%)", "3 (10-20%)", "4 (demais)"])
    partes.append(s)

lista = (pd.concat(partes, ignore_index=True)
           .sort_values(["etapa", "prob_calibrada"], ascending=[True, False]))
lista.to_csv("resultados/05_implantacao/lista_priorizada_escolas.csv", index=False, encoding="utf-8-sig")

(lista.groupby(["etapa", "faixa_prioridade"], observed=True)
      .agg(escolas=("codigo_escola", "size"), prob_media=("prob_calibrada", "mean"))
      .round(3))


In [ ]:
# Célula 10 · Relatório PDF da avaliação (versão 3: inclui células 13 a 16)
# Deve ser a ÚLTIMA célula do notebook. Requer reportlab (célula 1).
# Salve o notebook no GitHub e rode !git pull -q ANTES: o código no fim do PDF é lido de lá.

import os, json, datetime
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.units import cm
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import (SimpleDocTemplate, Paragraph, Spacer, Table,
                                TableStyle, Image, Preformatted, PageBreak)

PASTA = "resultados/04_avaliacao"
PASTA_IMPL = "resultados/05_implantacao"
PASTA_EDA = "resultados/02_eda"
PDF = os.path.join(PASTA, "relatorio_avaliacao_implantacao.pdf")
NOTEBOOK = "notebooks/04_avaliacao_implantacao.ipynb"

def ler(nome, **kw):
    return pd.read_csv(os.path.join(PASTA, nome), encoding="utf-8-sig", **kw)

def limpa(texto):
    return (str(texto).replace("→", "->").replace("×", "x")
            .replace("≥", ">=").replace("≤", "<="))

recall_k = ler("01_recall_at_k.csv")
calibracao = ler("02_calibracao.csv")
shap_tab = ler("03_shap_importancia.csv")
sensib = ler("04_sensibilidade_percentis.csv")
shap_ai = ler("05_shap_ai_p80_vs_p90.csv", index_col=0)
perfil_num = ler("06_perfil_numericas.csv")
perfil_cat = ler("07_perfil_categoricas.csv")
geo_uf = ler("08_geografia_uf.csv")
urb_rur = ler("09_alto_abandono_urbana_rural.csv")
arq_aus = os.path.join(PASTA_EDA, "12_ausencia_por_etapa.csv")
ausencia = pd.read_csv(arq_aus, encoding="utf-8-sig", index_col=0) if os.path.exists(arq_aus) else None

dobras = ler("14_metricas_por_dobra.csv")
estab = ler("16_estabilidade_importancia.csv")
erros = ler("17_analise_erros.csv")
municipios = ler("18_municipios.csv", dtype={"CO_MUNICIPIO": str})
lista = pd.read_csv(os.path.join(PASTA_IMPL, "lista_priorizada_escolas.csv"), encoding="utf-8-sig",
                    dtype={"codigo_escola": str, "CO_MUNICIPIO": str})

overfit = (dobras.groupby(["etapa", "parte"])[["pr_auc", "roc_auc", "acuracia", "f1"]]
                 .agg(["mean", "std"]).round(3))
overfit.columns = [f"{m}_{e}" for m, e in overfit.columns]
overfit = overfit.reset_index()

top10 = recall_k[recall_k["k_pct"] == 10][["etapa", "precisao", "acaso", "vezes_melhor_que_acaso"]].copy()
top10["meta_vezes_acaso"] = 2.0
top10["atende"] = top10["vezes_melhor_que_acaso"] >= top10["meta_vezes_acaso"]
PR_AUC_S5 = pd.DataFrame({
    "etapa": ["ensino_medio", "anos_finais", "anos_iniciais"],
    "modelo_escolhido": ["Random Forest", "Hist Gradient Boosting", "Hist Gradient Boosting"],
    "pr_auc_modelo": [0.5177, 0.5177, 0.4710],
    "pr_auc_reg_logistica": [0.4937, 0.4720, 0.4125],
})
PR_AUC_S5["atende"] = PR_AUC_S5["pr_auc_modelo"] > PR_AUC_S5["pr_auc_reg_logistica"]

fig, eixos = plt.subplots(1, 3, figsize=(12, 3.6), sharey=True)
for i, etapa in enumerate(recall_k["etapa"].unique()):
    d = recall_k[recall_k["etapa"] == etapa]
    rotulos = [f"Top {k}%" for k in d["k_pct"]]
    eixos[i].bar(rotulos, d["precisao"], color="#2b6cb0", label="Lista do modelo")
    eixos[i].axhline(d["acaso"].iloc[0], color="#dd6b20", linestyle="--", label="Acaso (prevalência)")
    for j, v in enumerate(d["precisao"]):
        eixos[i].text(j, v + 0.01, f"{v:.2f}", ha="center", fontsize=8)
    eixos[i].set_title(etapa.replace("_", " ").title(), fontsize=10, fontweight="bold")
    eixos[i].set_ylim(0, 0.8)
eixos[0].set_ylabel("Precisão na lista")
eixos[0].legend(fontsize=7, loc="upper right")
plt.tight_layout()
grafico_recall = os.path.join(PASTA, "10_recall_at_k.png")
plt.savefig(grafico_recall, dpi=200, bbox_inches="tight")
plt.close()

estilos = getSampleStyleSheet()
TITULO = ParagraphStyle("T", parent=estilos["Heading1"], fontSize=14, leading=18,
                        textColor=colors.HexColor("#1a202c"))
H2 = ParagraphStyle("H2", parent=estilos["Heading2"], fontSize=11, leading=14,
                    textColor=colors.HexColor("#2b6cb0"), spaceBefore=10)
CORPO = ParagraphStyle("C", parent=estilos["Normal"], fontSize=8.5, leading=12,
                       textColor=colors.HexColor("#2d3748"), spaceAfter=6)
CODIGO = ParagraphStyle("Cod", parent=estilos["Normal"], fontName="Courier",
                        fontSize=6.2, leading=8)

def tabela(df, larguras=None, max_linhas=40):
    df = df.head(max_linhas)
    cab = ParagraphStyle("cab", fontName="Helvetica-Bold", fontSize=6.5, leading=8,
                         textColor=colors.white)
    dados_tab = [[Paragraph(limpa(c).replace("_", " "), cab) for c in df.columns]] + \
                [[limpa(v) for v in linha] for linha in df.astype(str).values.tolist()]
    if larguras is None and len(df.columns) >= 7:
        larguras = [17.6 * cm / len(df.columns)] * len(df.columns)
    t = Table(dados_tab, colWidths=larguras, repeatRows=1)
    t.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#2b6cb0")),
        ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
        ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
        ("FONTSIZE", (0, 0), (-1, -1), 6.8),
        ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.HexColor("#edf2f7")]),
        ("GRID", (0, 0), (-1, -1), 0.25, colors.HexColor("#cbd5e0")),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
    ]))
    return t

def figura(caminho, largura=17 * cm):
    if not os.path.exists(caminho):
        return Paragraph(f"[Gráfico não encontrado: {caminho}]", CORPO)
    img = Image(caminho)
    img.drawHeight = largura * img.imageHeight / img.imageWidth
    img.drawWidth = largura
    return img

h = []
h.append(Paragraph("Relatório Técnico - Fases 5 e 6: Avaliação e Implantação", TITULO))
h.append(Paragraph(
    f"Gerado em {datetime.date.today():%d/%m/%Y}. Avaliação dos modelos vencedores do notebook 03 "
    "(Random Forest no ensino médio; Hist Gradient Boosting nos anos finais e iniciais), "
    "alvo P80 por etapa, validação cruzada agrupada por município (StratifiedGroupKFold, 5 dobras). "
    "Todas as métricas de desempenho usam probabilidades fora da dobra (escolas não vistas no treino). "
    "Os resultados descrevem associações no nível da escola e não indicam causalidade.", CORPO))

h.append(Paragraph("Resumo dos resultados", H2))
resumo = [
    "Lista dos 10% de maior risco: precisão de " + ", ".join(
        f"{r.precisao:.2f} ({r.vezes_melhor_que_acaso:.1f}x o acaso) em {r.etapa.replace('_', ' ')}"
        for r in top10.itertuples()) + ".",
    "Critério de aceite atendido em todas as etapas: " + ("sim" if top10["atende"].all() and PR_AUC_S5["atende"].all() else "não") + " (seção 9).",
    "Após calibração, a probabilidade média prevista coincide com a prevalência real; o Brier cai de "
    + "; ".join(f"{r.etapa.replace('_', ' ')}: {r.brier_antes:.3f} para {r.brier_depois:.3f}" for r in calibracao.itertuples()) + ".",
    "A distorção idade-série está entre as variáveis de maior importância nas três etapas; nos anos iniciais, "
    "o porte da escola lidera em P80 e perde importância em P90 (artefato do rótulo taxa > 0).",
    "O desempenho se mantém acima do acaso nos limiares P75, P80 e P90 (seção 4).",
]
for item in resumo:
    h.append(Paragraph("- " + item, CORPO))

h.append(Paragraph("1. Recall@k: utilidade da lista priorizada", H2))
h.append(Paragraph(
    "As escolas são ordenadas pela probabilidade prevista e a lista é cortada na capacidade de "
    "atendimento (5%, 10% ou 20% das escolas). Precisão = proporção de escolas da lista com alto "
    "abandono real; recall = proporção de todas as escolas de alto abandono capturadas pela lista.", CORPO))
h.append(tabela(recall_k))
h.append(Spacer(1, 6))
h.append(figura(grafico_recall))

h.append(Paragraph("2. Calibração das probabilidades", H2))
h.append(Paragraph(
    "O uso de class_weight='balanced' infla as probabilidades. Foi aplicada calibração de Platt "
    "(regressão logística sobre o logit da probabilidade), ajustada também por grupos de município. "
    "A calibração altera a escala, não a ordem: a lista priorizada é preservada.", CORPO))
h.append(tabela(calibracao))
h.append(Spacer(1, 6))
h.append(figura(os.path.join(PASTA, "02_calibracao.png")))

h.append(PageBreak())
h.append(Paragraph("3. Interpretação com SHAP", H2))
h.append(Paragraph(
    "Valores SHAP calculados no modelo final de cada etapa, em amostra de 2.000 escolas. "
    "Colunas one-hot foram somadas na variável original. A escala difere entre Random Forest "
    "(probabilidade) e Hist Gradient Boosting (log-odds): comparar a ordem, não os valores. "
    "A direção indica associação, não efeito causal.", CORPO))
h.append(figura(os.path.join(PASTA, "03_shap_importancia.png")))
h.append(Spacer(1, 6))
h.append(tabela(shap_tab.groupby("etapa").head(8), max_linhas=24))

h.append(Paragraph("4. Sensibilidade ao limiar (P75, P80, P90)", H2))
h.append(Paragraph(
    "A PR-AUC cai com o limiar porque a classe positiva fica menor; a comparação justa é o ganho "
    "sobre o acaso (pr_auc_vezes_acaso e vezes_acaso_top10). Nos anos iniciais, P75 e P80 têm "
    "limite zero e produzem o mesmo alvo.", CORPO))
h.append(tabela(sensib))

h.append(Paragraph("5. Anos iniciais: importância do porte em P80 e P90", H2))
h.append(Paragraph(
    "Teste do artefato do rótulo taxa > 0: se o porte perde importância quando o alvo exige taxa "
    "acima de 0,6% (P90), parte de seu peso em P80 decorre da maior chance de escolas grandes "
    "registrarem ao menos um abandono.", CORPO))
h.append(tabela(shap_ai.reset_index().rename(columns={"index": "variavel"}).round(4).head(8)))

h.append(PageBreak())
h.append(Paragraph("6. Perfil das escolas priorizadas (10% do topo)", H2))
h.append(Paragraph("Medianas das escolas priorizadas comparadas às demais.", CORPO))
h.append(tabela(perfil_num))
h.append(Spacer(1, 6))
h.append(Paragraph("Composição por localização e rede (% dentro de cada grupo).", CORPO))
h.append(tabela(perfil_cat))

h.append(Paragraph("7. Distribuição geográfica proporcional", H2))
h.append(Paragraph(
    "Razão de representação = participação da UF na lista priorizada dividida por sua "
    "participação no total de escolas da etapa (acima de 1 = acima do esperado pelo tamanho). "
    "O percentual real de alto abandono vem do dado, sem modelo. Como a UF é preditora, "
    "afirmações devem se apoiar no percentual real. Não constitui ranking de desempenho.", CORPO))
geo_top = (geo_uf.sort_values(["etapa", "razao_representacao"], ascending=[True, False])
                 .groupby("etapa").head(5)
                 [["etapa", "SG_UF", "escolas", "pct_escolas_da_etapa", "pct_das_prioritarias",
                   "razao_representacao", "pct_alto_abandono_real"]])
h.append(tabela(geo_top))
h.append(Spacer(1, 6))
h.append(Paragraph("Percentual de escolas com alto abandono (P80) por localização.", CORPO))
h.append(tabela(urb_rur))

h.append(PageBreak())
h.append(Paragraph("8. Análise exploratória complementar (feedback F2)", H2))
h.append(Paragraph(
    "Matriz de correlação de Spearman entre as preditoras contínuas e a taxa de abandono, por etapa. "
    "Pares acima de 0,7 indicam redundância (porte, alunos por turma e alunos por docente; "
    "horas-aula e tempo integral).", CORPO))
h.append(figura(os.path.join(PASTA_EDA, "03_matriz_correlacao.png")))
h.append(Spacer(1, 6))
h.append(Paragraph(
    "Distribuição das preditoras por etapa (1% superior omitido apenas para legibilidade).", CORPO))
h.append(figura(os.path.join(PASTA_EDA, "04_histogramas_preditoras.png"), largura=16 * cm))
if ausencia is not None:
    h.append(Paragraph(
        "Percentual de valores ausentes por variável e etapa. A ausência nos anos iniciais concentra-se "
        "em escolas sem turma registrada da etapa; a imputação ocorre dentro do pipeline.", CORPO))
    h.append(tabela(ausencia.reset_index().rename(columns={"index": "variavel"})))

h.append(Paragraph("9. Critério de aceite (feedback F3)", H2))
h.append(Paragraph(
    "Fixado após o modelo de referência, como previsto no relatório das Etapas 1 e 2: "
    "(1) precisão na lista dos 10% de maior risco de pelo menos 2 vezes a prevalência; "
    "(2) PR-AUC do modelo escolhido maior que a da Regressão Logística na validação agrupada "
    "por município (valores do notebook 03).", CORPO))
h.append(tabela(top10))
h.append(Spacer(1, 6))
h.append(tabela(PR_AUC_S5))

h.append(PageBreak())
h.append(Paragraph("10. Diagnóstico de overfitting e variação entre dobras", H2))
h.append(Paragraph(
    "Métricas no treino e na validação de cada uma das 5 dobras agrupadas por município (média e "
    "desvio-padrão). A diferença entre treino e validação mede o quanto o modelo se ajusta às escolas "
    "vistas; o desvio entre dobras mede a estabilidade. Acurácia e F1 usam corte de 0,5.", CORPO))
h.append(tabela(overfit))
h.append(Spacer(1, 6))
h.append(Paragraph(
    "Estabilidade das variáveis: importância por permutação (queda da PR-AUC ao embaralhar a variável) "
    "em cada dobra; posição mínima e máxima entre as 5 dobras.", CORPO))
h.append(tabela(estab.groupby("etapa").head(5)))

h.append(Paragraph("11. Análise de erros (lista dos 10%)", H2))
h.append(Paragraph(
    "Medianas por tipo de resultado. Falso positivo: escola na lista sem alto abandono registrado. "
    "Falso negativo: escola com alto abandono fora da lista.", CORPO))
h.append(tabela(erros[["etapa", "tipo"] + [c for c in erros.columns if c not in ("etapa", "tipo")]]
                  .rename(columns=lambda c: c.replace("_distorcao", "").replace("_formacao_adequada", ""))))

h.append(Paragraph("12. Municípios (cálculo corrigido)", H2))
h.append(Paragraph(
    "Percentual de escolas com alto abandono real e percentual na lista priorizada, por município e "
    "etapa, entre municípios com pelo menos 20 escolas na etapa. Numerador e denominador contam as "
    "mesmas linhas escola x etapa. Não constitui ranking de desempenho.", CORPO))
h.append(tabela(municipios[municipios["escolas"] >= 20]
                .sort_values(["etapa", "pct_alto_abandono_real"], ascending=[True, False])
                .groupby("etapa").head(5)))

h.append(Paragraph("13. Implantação: lista priorizada de escolas", H2))
h.append(Paragraph(
    "Arquivo resultados/05_implantacao/lista_priorizada_escolas.csv: uma linha por escola x etapa, "
    "com probabilidade calibrada e faixa de prioridade. Uso previsto: apoio à priorização de ações de "
    "permanência (busca ativa, apoio pedagógico), nunca como avaliação ou punição da escola.", CORPO))
h.append(tabela(lista.groupby(["etapa", "faixa_prioridade"], observed=True)
                     .agg(escolas=("codigo_escola", "size"), prob_media=("prob_calibrada", "mean"))
                     .round(3).reset_index()))
h.append(Spacer(1, 6))
h.append(Paragraph("Exemplo: primeiras escolas da lista em cada etapa.", CORPO))
h.append(tabela(lista.groupby("etapa").head(3)))

if os.path.exists(NOTEBOOK):
    with open(NOTEBOOK, encoding="utf-8") as f:
        celulas = json.load(f)["cells"]
    codigo = "\n\n".join("".join(c["source"]) for c in celulas if c["cell_type"] == "code")
    h.append(PageBreak())
    h.append(Paragraph("14. Código executado (notebook 04_avaliacao)", H2))
    h.append(Preformatted(limpa(codigo), CODIGO, maxLineLength=150))

doc = SimpleDocTemplate(PDF, pagesize=A4, leftMargin=1.6 * cm, rightMargin=1.6 * cm,
                        topMargin=1.5 * cm, bottomMargin=1.5 * cm)
doc.build(h)
print("[OK] Relatório salvo em", PDF)


## Baixar os resultados
Compacta as pastas de saída para subir no GitHub (Add file → Upload files).

In [ ]:
!zip -qr /content/resultados_04_05.zip resultados/04_avaliacao resultados/05_implantacao
from google.colab import files
files.download("/content/resultados_04_05.zip")
